# OpenSSH Log Regex and Collections Exercise

Complete all five tasks within 75 minutes. Use the `re` module and Python collections. Do not change the supplied file path, input data, function names, return formats, or assertions.

Each task is complete when its assertion cell prints `Task n passed`. Functions must not modify `log_lines`.

## Load the OpenSSH Log

The file is read into a list. Each list element contains one log line without its trailing newline character.

In [ ]:
from pathlib import Path
import re

log_path = Path(r"C:\data\loghub\openssh\openssh_2k.log")

with log_path.open(mode="r", encoding="utf-8") as log_file:
    log_lines = [line.rstrip("\n") for line in log_file]

print(f"Loaded {len(log_lines):,} lines from {log_path}")
print(log_lines[0])

In [ ]:
assert isinstance(log_lines, list)
assert len(log_lines) == 2000
assert all(isinstance(line, str) for line in log_lines)
assert log_lines[0].startswith("Dec 10 06:55:46 LabSZ sshd[24200]:")
assert log_lines[-1].endswith("Failed password for invalid user user from 103.99.0.122 port 52683 ssh2")
print("Log input validated")

## 1. Parse Every Log Line — 15 minutes

Complete `parse_log_lines(lines)`.

Use one compiled regex pattern with named groups to parse this common format:

```text
Dec 10 06:55:46 LabSZ sshd[24200]: message text
```

Return a list of tuples. Each tuple must contain:

```text
(timestamp, host, process_id, message)
```

Requirements:

- `timestamp` must be a string such as `Dec 10 06:55:46`.
- `host` and `message` must be strings.
- `process_id` must be an integer.
- Raise `ValueError("Unrecognized log line: <line>")` when a line does not match.
- Preserve input order.
- Do not modify `lines`.

In [ ]:
def parse_log_lines(lines):
    # Write your regex and collection processing here
    pass

In [ ]:
line_snapshot = log_lines.copy()
parsed_logs = parse_log_lines(log_lines)

assert isinstance(parsed_logs, list)
assert len(parsed_logs) == 2000
assert parsed_logs[0] == (
    "Dec 10 06:55:46",
    "LabSZ",
    24200,
    "reverse mapping checking getaddrinfo for ns.marryaldkfaczcz.com [173.234.31.186] failed - POSSIBLE BREAK-IN ATTEMPT!",
)
assert parsed_logs[-1] == (
    "Dec 10 11:04:45",
    "LabSZ",
    25539,
    "Failed password for invalid user user from 103.99.0.122 port 52683 ssh2",
)
assert all(isinstance(record, tuple) and len(record) == 4 for record in parsed_logs)
assert all(isinstance(record[2], int) for record in parsed_logs)
assert log_lines == line_snapshot

try:
    parse_log_lines(["not an ssh log line"])
    raise AssertionError("An invalid log line was accepted")
except ValueError as error:
    assert str(error) == "Unrecognized log line: not an ssh log line"

print("Task 1 passed")

## 2. Extract Failed Password Events — 15 minutes

Complete `extract_failed_passwords(lines)`. Match both of these message formats:

```text
Failed password for root from 183.62.140.253 port 36060 ssh2
Failed password for invalid user webmaster from 173.234.31.186 port 38926 ssh2
message repeated 5 times: [ Failed password for root from 5.36.59.76 port 42393 ssh2]
```

Return a list of tuples in input order. Each tuple must contain:

```text
(username, ip_address, port, is_invalid_user)
```

Requirements:

- `username` and `ip_address` must be strings.
- `port` must be an integer.
- `is_invalid_user` must be a Boolean.
- Match the complete message rather than a partial message.
- Accept one or more spaces after `invalid user`.
- Accept the optional `message repeated <number> times: [ ... ]` wrapper as one log entry; do not expand its repetition count.
- Ignore every non-matching event.
- Do not modify `lines`.

In [ ]:
def extract_failed_passwords(lines):
    # Write your regex and collection processing here
    pass

In [ ]:
line_snapshot = log_lines.copy()
failed_passwords = extract_failed_passwords(log_lines)

assert isinstance(failed_passwords, list)
assert len(failed_passwords) == 520
assert failed_passwords[0] == ("webmaster", "173.234.31.186", 38926, True)
assert failed_passwords[-1] == ("user", "103.99.0.122", 52683, True)
assert sum(event[3] for event in failed_passwords) == 135
assert sum(not event[3] for event in failed_passwords) == 385
assert all(isinstance(event, tuple) and len(event) == 4 for event in failed_passwords)
assert all(isinstance(event[2], int) and isinstance(event[3], bool) for event in failed_passwords)
assert log_lines == line_snapshot
assert extract_failed_passwords(["Failed password text without required fields"]) == []
print("Task 2 passed")

## 3. Summarize Failed Authentication — 15 minutes

Complete `summarize_failures(failed_events)`. Use the tuples produced by Task 2.

Return a dictionary with exactly these entries:

- `user_counts`: dictionary mapping username to failed-attempt count.
- `ip_counts`: dictionary mapping IP address to failed-attempt count.
- `unique_users`: set of usernames.
- `unique_ips`: set of IP addresses.
- `top_users`: list of the five `(username, count)` tuples with the highest counts.
- `top_ips`: list of the five `(ip_address, count)` tuples with the highest counts.

Sort both top-five lists by count descending and then name or IP ascending when counts are equal. Do not modify `failed_events`.

In [ ]:
def summarize_failures(failed_events):
    # Write your collection processing here
    pass

In [ ]:
failed_snapshot = failed_passwords.copy()
failure_summary = summarize_failures(failed_passwords)

assert set(failure_summary) == {
    "user_counts", "ip_counts", "unique_users", "unique_ips", "top_users", "top_ips"
}
assert isinstance(failure_summary["user_counts"], dict)
assert isinstance(failure_summary["ip_counts"], dict)
assert isinstance(failure_summary["unique_users"], set)
assert isinstance(failure_summary["unique_ips"], set)
assert len(failure_summary["unique_users"]) == 63
assert len(failure_summary["unique_ips"]) == 23
assert sum(failure_summary["user_counts"].values()) == 520
assert sum(failure_summary["ip_counts"].values()) == 520
assert failure_summary["top_users"] == [
    ("root", 370),
    ("admin", 44),
    ("oracle", 6),
    ("support", 6),
    ("test", 5),
]
assert failure_summary["top_ips"] == [
    ("183.62.140.253", 286),
    ("187.141.143.180", 80),
    ("103.99.0.122", 46),
    ("112.95.230.3", 26),
    ("5.188.10.180", 18),
]
assert failed_passwords == failed_snapshot
print("Task 3 passed")

## 4. Correlate Invalid Users and Break-In Warnings — 15 minutes

Complete `correlate_suspicious_events(lines)`. Use separate compiled regex patterns for these messages:

```text
Invalid user webmaster from 173.234.31.186
reverse mapping checking getaddrinfo for ns.example.com [173.234.31.186] failed - POSSIBLE BREAK-IN ATTEMPT!
```

Return a dictionary containing:

- `invalid_user_events`: list of `(username, ip_address)` tuples, including repeated events.
- `invalid_users`: set of distinct usernames.
- `invalid_user_ips`: set of distinct IP addresses from invalid-user events.
- `breakin_events`: list of `(hostname, ip_address)` tuples, including repeated events.
- `breakin_endpoints`: set of distinct `(hostname, ip_address)` tuples.
- `common_ips`: set of IP addresses present in both event types.

Preserve input order in both lists. Do not modify `lines`.

In [ ]:
def correlate_suspicious_events(lines):
    # Write your regex and collection processing here
    pass

In [ ]:
line_snapshot = log_lines.copy()
suspicious = correlate_suspicious_events(log_lines)

assert len(suspicious["invalid_user_events"]) == 112
assert len(suspicious["invalid_users"]) == 56
assert len(suspicious["invalid_user_ips"]) == 19
assert len(set(suspicious["invalid_user_events"])) == 76
assert suspicious["invalid_user_events"][0] == ("webmaster", "173.234.31.186")
assert len(suspicious["breakin_events"]) == 85
assert suspicious["breakin_events"][0] == ("ns.marryaldkfaczcz.com", "173.234.31.186")
assert suspicious["breakin_endpoints"] == {
    ("ns.marryaldkfaczcz.com", "173.234.31.186"),
    ("195-154-37-122.rev.poneytelecom.eu", "195.154.37.122"),
    ("191-210-223-172.user.vivozap.com.br", "191.210.223.172"),
    ("customer-187-141-143-180-sta.uninet-ide.com.mx", "187.141.143.180"),
}
assert suspicious["common_ips"] == {
    "173.234.31.186", "187.141.143.180", "195.154.37.122"
}
assert log_lines == line_snapshot
print("Task 4 passed")

## 5. Build a Security Report — 15 minutes

Complete `build_security_report(lines, minimum_failures=10)`. Process the original log lines with regex.

Return a dictionary containing:

- `total_lines`: total number of lines.
- `failed_passwords`: number of `Failed password` events.
- `authentication_failures`: number of PAM `pam_unix(sshd:auth): authentication failure;` events.
- `invalid_user_events`: number of `Invalid user <name> from <ip>` events.
- `breakin_warnings`: number of `POSSIBLE BREAK-IN ATTEMPT!` events.
- `accepted_logins`: list of `(method, username, ip_address, port)` tuples. Convert the port to an integer.
- `unique_ipv4_addresses`: set of every IPv4 address found anywhere in the file.
- `suspicious_ips`: set of IPs with at least `minimum_failures` failed-password events.
- `top_attacker`: tuple `(ip_address, failed_password_count)`. Sort by IP ascending to break a count tie. Return `None` when no failed-password event exists.

Use regex for event recognition and field extraction. Do not modify `lines`.

In [ ]:
def build_security_report(lines, minimum_failures=10):
    # Write your regex and collection processing here
    pass

In [ ]:
line_snapshot = log_lines.copy()
report = build_security_report(log_lines)

assert report["total_lines"] == 2000
assert report["failed_passwords"] == 520
assert report["authentication_failures"] == 494
assert report["invalid_user_events"] == 112
assert report["breakin_warnings"] == 85
assert report["accepted_logins"] == [("password", "fztu", "119.137.62.142", 49116)]
assert isinstance(report["unique_ipv4_addresses"], set)
assert len(report["unique_ipv4_addresses"]) == 30
assert report["suspicious_ips"] == {
    "5.188.10.180",
    "103.99.0.122",
    "112.95.230.3",
    "183.62.140.253",
    "185.190.58.151",
    "187.141.143.180",
}
assert report["top_attacker"] == ("183.62.140.253", 286)
assert build_security_report([], minimum_failures=1) == {
    "total_lines": 0,
    "failed_passwords": 0,
    "authentication_failures": 0,
    "invalid_user_events": 0,
    "breakin_warnings": 0,
    "accepted_logins": [],
    "unique_ipv4_addresses": set(),
    "suspicious_ips": set(),
    "top_attacker": None,
}
assert log_lines == line_snapshot
print("Task 5 passed")

## Completion Check

Restart the kernel and run all cells from the beginning. The notebook is complete when `Log input validated` and `Task 1 passed` through `Task 5 passed` appear without an exception.

The completed work must contain lists, tuples, sets, and dictionaries, with regex responsible for recognizing and extracting log fields.